# 03 · Train the codec-robust model  (P2) ★

**The contribution.**

Identical to notebook 02 in every respect except the feature cache it reads.
Keeping the difference to a single argument is what makes the comparison
meaningful — anything else that varied would confound the result.

The deliverable of this phase is not a model. It is a **comparison**.

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Run this first in every notebook.  Idempotent.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Keep the repo and all caches on Drive so a disconnect does not cost you
    # the feature extraction pass.
    PROJECT = Path("/content/drive/MyDrive/voice-integrity")
    if not PROJECT.exists():
        raise SystemExit(
            f"Upload or clone the repo to {PROJECT} first.\n"
            "  !git clone <your-repo-url> /content/drive/MyDrive/voice-integrity"
        )
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

# Repo-local model cache.  Set BEFORE importing transformers, or it will use
# the default location and the cache will not be portable to the demo machine.
os.environ["HF_HOME"] = str(PROJECT / "cache" / "huggingface")
os.environ["TORCH_HOME"] = str(PROJECT / "cache" / "torch")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("project:", PROJECT)
print("python :", sys.version.split()[0])

In [ ]:
if IN_COLAB:
    !pip install -q transformers speechbrain soundfile librosa pydantic pyyaml cryptography wandb
    !apt-get -qq install -y ffmpeg libopencore-amrnb-dev > /dev/null

# AMR-NB encoding is the one that silently goes missing.  If this prints
# nothing, your mobile-codec augmentation does nothing and the whole
# codec-robustness result quietly evaporates.
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -i amr || echo "AMR-NB ENCODER MISSING"

In [ ]:
import torch
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         :", torch.cuda.get_device_name(0))
    print("memory         : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Why this works

A phone call is 8 kHz with a roughly 300–3400 Hz passband. More than half the
spectrum is discarded, and the synthesis artifacts a detector relies on are
concentrated in exactly the part that goes.

Published systems fall from about 1% EER on clean audio to 15–25% through
codecs. Training on the degraded distribution closes that gap — the forger did
not hide the evidence, the transmission did.

In [ ]:
from vif.common.config import load_config
from vif.data.manifests import read_manifest
from vif.models.heads import build_head

config = load_config("configs")

# NOTE the only change from notebook 02: the codec-degraded caches.
train_items = read_manifest("data/features/train_codec/manifest.jsonl")
dev_items   = read_manifest("data/features/dev_codec/manifest.jsonl")

head = build_head(config.model.head, feat_dim=config.model.frontend.hidden_dim)

In [ ]:
from vif.train.loop import TrainConfig, train_head

# Identical hyperparameters to the baseline.  Do not tune these separately.
train_config = TrainConfig(
    epochs=25, batch_size=32, learning_rate=1e-4,
    class_weighting=True, early_stop_patience=6,
)

history = train_head(
    head,
    train_items, dev_items,
    train_features="data/features/train_codec",
    dev_features="data/features/dev_codec",
    train_config=train_config,
    device=DEVICE,
    checkpoint_path="models/checkpoints/codec_robust.pt",
    checkpoint_meta={
        "arch": config.model.head.arch,
        "feat_dim": config.model.frontend.hidden_dim,
        "frontend_id": config.model.frontend.model_id,
        "window_samples": config.model.audio.window_samples,
        "condition": "codec_robust",
    },
)
print(f"\nbest dev EER {history.best_eer*100:.2f}% at epoch {history.best_epoch+1}")

## Calibrate this model too

Each model needs its own calibration. This one is the model the server loads,
so its constants go to `configs/calibration.json`; reusing the baseline's would
turn the served probability into a number fitted to a different model.

In [ ]:
from torch.utils.data import DataLoader
from vif.data.datasets import FeatureDataset, collate_features
from vif.eval.calibration import Calibrator, fit_platt
from vif.train.loop import predict

dev_loader = DataLoader(
    FeatureDataset("data/features/dev_codec", dev_items, max_frames=208),
    batch_size=32, shuffle=False, collate_fn=collate_features,
)
labels, scores = predict(head, dev_loader, DEVICE)
params = fit_platt(labels, scores, branch="spoof", split="dev")
Calibrator({"spoof": params}).save("configs/calibration.json")
print(f"llr = {params.a:.4f} * score + {params.b:.4f}")

## Exit criteria for P2

The baseline collapses toward chance on codec-degraded audio while this model
holds. That single chart — produced in notebook 04 — **is** the presentation.

> A complete, defensible submission exists at this point. Nothing later is
> worth cutting into P0–P2 to reach.